# napari

napari is an open-source, Python-based, multi-dimensional image viewer

It supports interactive data inspection, annotation, and plugin-based extensions.

[GitHub](https://github.com/napari/napari) - [Docs](https://napari.org/stable/index.html)


---

**Plugin Ecosystem**

napari can be extended through community-developed plugins that add new functionality

NOTE: You can discover napari plugins at [napari-hub](https://napari-hub.org/)

---

**Running napari**

You can start napari GUI:
1. from terminal
    ```bash
    napari
    ```

2. from Python notebook 
    ```python
    import napari
    viewer = napari.Viewer() # The viewer object provides programmatic access to the napari window (GUI)
    ```

---

## empanada-napari (30 min)

**Empanada** is a tool for organelle segmentation in 2D and 3D electron microscopy images using deep learning models.

It is integrated in napari as a plugin and is designed for interactive use inside the napari GUI.

[GitHub](https://github.com/volume-em/empanada-napari) - [Docs](https://empanada.readthedocs.io/en/latest/index.html)

<br>

Example data sources: [luchi_pp](https://www.ebi.ac.uk/empiar/EMPIAR-10982/), empanada demo datasets

<div style="border-left: 4px solid #4CAF50; padding-left: 1em;">

**TASK**: Explore deep-learning segmentation in napari using *empanada-napari*

*2D*
- Open napari
- Open the image **TEM_1.tiff** from the folder `empanada_example`
- Start 2D inference from empanada plugin menu
- Run a pre-trained empanada model: `MitoNet_v1`
- Test different parameters in the plugin menu
- Try other pre-trained models (e.g. `NucleoNet`)
- Test 2D inference on the image **mouse_liver_roi.tif** 

*3D*
- Remove all existing layers in napari
- Open the image **lucchi_pp_3d.tif** from the folder `napari_example`
- Start 3D inference from empanada plugin menu
- Run a pre-trained empanada model: `MitoNet_v1_mini`


</div>

NOTES:

- You can re-train (fine-tune) the model on your own dataset, or train a new panoptic segmentation model - [manual](https://empanada.readthedocs.io/en/latest/tutorials/train_panop.html#train-panoptic-model).
- The panoptic segmentation approach combines semantic and instance segmentation. 
- In **Empanada**, distinguishing classes and instances within the same mask is handled using label offsets (divisors) - e.g. [100, 200, 300..] define the semantic class and [101, 102, 201…] represent individual instances within given class.
- Unlike tools such as Cellpose, Empanada does not provide a strong standalone API workflow for batch processing outside the GUI.

In [ ]:
# Your code here



---

### **(Optional) Bonus** - ✨🧙 Custom widgets 🧙✨

When you want interactivity that is e.g. not part of the plugin you are using.

This example demonstrates how to create an interactive object filtering tool in `napari` using `magicgui`.

The `@magicgui` decorator automatically turns a Python function into a widget — in this case a slider widget - that lets you adjust the filter range dynamically.

In [ ]:
from magicgui import magicgui
import numpy as np
from skimage.measure import regionprops

# magicgui decorator for creating widget
@magicgui(
    call_button="Filter by solidity",
    range={"widget_type": "FloatRangeSlider", "min": 0.0, "max": 1.0, "step": 0.01},
)
def filter_by_solidity(
    viewer: napari.Viewer, # connecting current active viewer
    labels_layer: napari.layers.Labels, # dropdown of Labels layers from viewer
    range=(0.5, 1.0), # initial slider range
):

    # read labels layer array
    labels = np.asarray(labels_layer.data)
    
    # measure properties (including solidity)
    props = regionprops(labels)

    # filter objects based on selected solidity range
    keep_ids = [p.label for p in props if range[0] <= p.solidity <= range[1]]
    # create filtered labels array
    mask = np.isin(labels, keep_ids)
    filtered = labels * mask

    # create or update filtered labels layer in napari
    new_lables_name = f"{labels_layer.name}_solidity_filtered"
    if new_lables_name in viewer.layers:
        viewer.layers[new_lables_name].data = filtered
    else:
        viewer.add_labels(filtered, name=new_lables_name)

# add widget to viewer
viewer.window.add_dock_widget(filter_by_solidity, area="right")